# 02 — Preprocessing & Label Conversion
Class filtering, stratified subsetting, train/val/test split, BDD100K → YOLO format conversion, and visual verification.

In [ ]:
import shutil
import os

source_dir = '/kaggle/input/datasets/bddk100k-object-detection/Object-Detection'
destination_dir = '/kaggle/working/bdd100k_project'

if os.path.exists(source_dir):
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
    print("Files copied to /kaggle/working/src and ready to use!")
else:
    print("Source directory not found. Check your paths!")

In [ ]:
!pip install ultralytics --no-deps

In [ ]:
import sys
import os
import numpy as np

project_path = "/kaggle/working/bdd100k_project"
if project_path not in sys.path:
    sys.path.append(project_path)
    sys.path.append(os.path.join(project_path, 'src'))

from src.utils import seed_everything, log_environment

seed_everything()
log_environment()

In [ ]:
import json
import os
import glob
import random

DATASET_PATH = "/kaggle/input/datasets/maingochoang/bdd100k"
TRAIN_JSON = os.path.join(DATASET_PATH, "Bdd100k/bdd100k_labels_images_train.json")
VAL_JSON = os.path.join(DATASET_PATH, "Bdd100k/bdd100k_labels_images_val.json")
TRAIN_IMAGE_DIR = os.path.join(DATASET_PATH, "Bdd100k/train")
VAL_IMAGE_DIR = os.path.join(DATASET_PATH, "Bdd100k/val")

OUTPUT_DIR = "/kaggle/working/bdd100k_project/bdd100k-yolo-subset-v1"
LABEL_OUTPUT_DIR = "/kaggle/working/bdd100k_project/yolo_labels_temp"
RESULTS_PLOTS = "/kaggle/working/bdd100k_project/results/plots"
os.makedirs(RESULTS_PLOTS, exist_ok=True)

print("Done")

## Step 1: Load Annotations

In [ ]:
with open(TRAIN_JSON) as f:
    train_annotations = json.load(f)
with open(VAL_JSON) as f:
    val_annotations = json.load(f)

all_annotations = train_annotations + val_annotations
print(f"Total annotations loaded: {len(all_annotations)}")

## Step 2: Class Filtering
Remove images that contain no instances of our 8 selected classes.

In [ ]:
from src.preprocess import filter_relevant_images

filtered = filter_relevant_images(all_annotations)

## Step 3: Stratified Subsetting
Sample a subset preserving weather × timeofday proportions.

In [ ]:
from src.preprocess import stratified_subset

TRAIN_SIZE = 7000
VAL_SIZE = 1500
TEST_SIZE = 1500

train_items, val_items, test_items = stratified_subset(
    filtered, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE
)

## Step 4: Convert BDD100K Labels → YOLO Format

In [ ]:
import shutil
import os

# 1. Clear the label output directory
if os.path.exists(LABEL_OUTPUT_DIR):
    print(f"Cleaning up labels at: {LABEL_OUTPUT_DIR}")
    shutil.rmtree(LABEL_OUTPUT_DIR)
os.makedirs(LABEL_OUTPUT_DIR, exist_ok=True)

old_jsons = [
    "/kaggle/working/train_annotations.json", 
    "/kaggle/working/val_annotations.json", 
    "/kaggle/working/test_annotations.json",
    "/kaggle/working/bdd100k_project/filtered_annotations.json"
]

for json_file in old_jsons:
    if os.path.exists(json_file):
        os.remove(json_file)

print("Cleanup complete. New Fresh Run")

In [ ]:
from src.convert_labels import convert_bdd100k_to_yolo

all_items = train_items + val_items + test_items

filtered_json_path = "/kaggle/working/bdd100k_project/filtered_annotations.json"
with open(filtered_json_path, "w") as f:
    json.dump(all_items, f)

all_image_dir = "/kaggle/working/bdd100k_project/all_images_symlink"
os.makedirs(all_image_dir, exist_ok=True)

if not os.path.exists(os.path.join(all_image_dir, "train")):
    os.symlink(TRAIN_IMAGE_DIR, os.path.join(all_image_dir, "train"))
if not os.path.exists(os.path.join(all_image_dir, "val")):
    os.symlink(VAL_IMAGE_DIR, os.path.join(all_image_dir, "val"))

class ItemImageDir:
    def __init__(self, train_dir, val_dir):
        self.train_dir = train_dir
        self.val_dir = val_dir
    def get(self, name):
        if os.path.exists(os.path.join(self.train_dir, name)):
            return self.train_dir
        return self.val_dir

img_dirs = ItemImageDir(TRAIN_IMAGE_DIR, VAL_IMAGE_DIR)

for split_name, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
    split_json = f"/kaggle/working/{split_name}_annotations.json"
    with open(split_json, "w") as f:
        json.dump(items, f)

    print(f"\nConverting {split_name} split...")
    convert_bdd100k_to_yolo(
        json_path=split_json,
        image_dirs=[TRAIN_IMAGE_DIR, VAL_IMAGE_DIR],
        output_dir=os.path.join(LABEL_OUTPUT_DIR, split_name),
    )

In [ ]:
for split in ['train', 'val', 'test']:
    path = os.path.join(LABEL_OUTPUT_DIR, split)
    if os.path.exists(path):
        print(f"{split.capitalize()} labels: {len(os.listdir(path))}")

## Step 5: Organize into YOLO Dataset Structure

In [ ]:
import shutil

for split_name, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
    img_dst = os.path.join(OUTPUT_DIR, "images", split_name)
    lbl_dst = os.path.join(OUTPUT_DIR, "labels", split_name)
    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(lbl_dst, exist_ok=True)

    paired_count = 0
    for item in items:
        img_name = item["name"]
        label_name = img_name.replace(".jpg", ".txt")
        
        # Determine source paths
        src_img = os.path.join(img_dirs.get(img_name), img_name)
        src_label = os.path.join(LABEL_OUTPUT_DIR, split_name, label_name)

        # Only copy and count if both files actually exist
        if os.path.exists(src_img) and os.path.exists(src_label):
            shutil.copy2(src_img, os.path.join(img_dst, img_name))
            shutil.copy2(src_label, os.path.join(lbl_dst, label_name))
            paired_count += 1

    print(f"{split_name}: successfully copied {paired_count} image/label pairs")

## Step 6: Post-Conversion Visual Verification

In [ ]:
from src.convert_labels import visual_verification

train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
train_lbl_dir = os.path.join(OUTPUT_DIR, "labels", "train")

train_images = glob.glob(os.path.join(train_img_dir, "*.jpg"))
print(f"Found {len(train_images)} training images for verification")

visual_verification(
    image_paths=train_images,
    label_dir=train_lbl_dir,
    output_path=os.path.join(RESULTS_PLOTS, "conversion_verification.png"),
    grid_size=3,
    num_samples=9,
)

## Step 7: Save as Kaggle Dataset

In [ ]:
for split_name in ["train", "val", "test"]:
    img_count = len(glob.glob(os.path.join(OUTPUT_DIR, "images", split_name, "*.jpg")))
    lbl_count = len(glob.glob(os.path.join(OUTPUT_DIR, "labels", split_name, "*.txt")))
    print(f"{split_name}: {img_count} images, {lbl_count} labels")

print("\nDataset ready at:", OUTPUT_DIR)
print("Save this directory as a Kaggle Dataset: bdd100k-yolo-subset-v1")

In [ ]:
import zipfile
import os

# Zip up the files if running inside kaggle or colab
output_zip = "/kaggle/working/bdd100k_yolo_labels.zip"
output_dir = OUTPUT_DIR 
extra_files = [
    "/kaggle/working/test_annotations.json",
    "/kaggle/working/train_annotations.json",
    "/kaggle/working/val_annotations.json"
]

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, start=output_dir)
            zipf.write(file_path, arcname=os.path.join(os.path.basename(output_dir), arcname))
    
    for file in extra_files:
        zipf.write(file, os.path.basename(file))

print(output_zip)